# Auto-Generating Aliases (`alias_generator`)

In modern architectures, APIs typically communicate in JSON (`camelCase`), but backend Python code should strictly follow PEP 8 (`snake_case`).

Manually assigning `Field(alias="...")` to dozens of fields across dozens of models is tedious and error-prone. Pydantic solves this by allowing you to define an **Alias Generator function** at the model configuration level.

## 1. Using Built-in Generators

Pydantic provides highly optimized utility functions to handle common naming conversions (like `to_camel`, `to_snake`, and `to_pascal`). You can pass these directly into the `model_config`.

```python
from pydantic import BaseModel, ConfigDict
from pydantic.alias_generators import to_camel

class Employee(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel)
    
    first_name: str
    last_name: str
    department_id: int

# The aliases are generated automatically!
e = Employee.model_validate_json('{"firstName": "John", "lastName": "Doe", "departmentId": 5}')
print(e.first_name) # John

```

## 2. Writing a Custom Alias Generator

The most common friction point between JSON and Python is the `id` field. In JSON, every object usually has an `id`. In Python, `id()` is a built-in function, so using it as a variable name triggers linter warnings (e.g., in Pylint or Flake8).

The standard Python convention is to append a trailing underscore: `id_`. However, Pydantic's `to_camel` function will simply generate `id_` as the alias, which fails to map to the JSON `"id"`.

We can write a custom alias generator that converts to camel case *and* strips trailing underscores!

```python
from pydantic import BaseModel, ConfigDict
from pydantic.alias_generators import to_camel

# 1. Define the custom generator function
def custom_alias_generator(field_name: str) -> str:
    # Convert to camelCase (e.g., "first_name" -> "firstName")
    camel_case = to_camel(field_name)
    # Strip the trailing underscore if it exists (e.g., "id_" -> "id")
    return camel_case.removesuffix('_')

# 2. Apply it to the model
class SafeModel(BaseModel):
    model_config = ConfigDict(alias_generator=custom_alias_generator)
    
    id_: int        # Automatically aliased to "id"
    list_: list     # Automatically aliased to "list"
    first_name: str # Automatically aliased to "firstName"

# 3. Test it out!
json_payload = '{"id": 101, "list": ["a", "b"], "firstName": "Alice"}'
model = SafeModel.model_validate_json(json_payload)

print(model.id_) # 101

```

## 3. Overriding the Generator

If you have a model with an `alias_generator`, but one specific field in your payload deviates from the pattern, you don't need to write complex `if/else` logic in your generator function.

You can simply apply a manual `Field(alias="...")` to that specific field. **Manual field aliases always override the `alias_generator`.**

```python
class OverrideModel(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel)
    
    first_name: str 
    last_name: str
    
    # The JSON payload sends this weirdly formatted key
    # It overrides the generator completely.
    ssn: int = Field(alias="SocialSecurityNum") 

```

### Interview Preparation: Auto-Generating Aliases

<details>
<summary><b>1. Your team wants to enforce `snake_case` in your Python codebase, but the frontend React team requires JSON payloads in `camelCase`. What is the cleanest, most scalable way to handle this using Pydantic?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
I would create a base class that inherits from `pydantic.BaseModel` and set its `model_config` to use `alias_generator=to_camel` (imported from `pydantic.alias_generators`). I would also configure `populate_by_name=True` so we can instantiate the models internally using standard Python names. All other models in the codebase would inherit from this base class, ensuring a completely automated, zero-touch translation between internal `snake_case` and external `camelCase`.
</details>

<details>
<summary><b>2. You define a model with `model_config = ConfigDict(alias_generator=to_camel)`. You have a field `department_id: int = Field(alias="DEPT_ID")`. When you call `model_dump(by_alias=True)`, what key will be output for that field? Why?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
The output key will be `"DEPT_ID"`. <br><br>
In Pydantic, explicit per-field configurations always take precedence over global model configurations. Because you manually defined the alias on the field, Pydantic ignores the `alias_generator` for that specific field.
</details>

<details>
<summary><b>3. Why is it considered a bad practice in Python to define a Pydantic field simply as `id: int`, and what is the standard workaround?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
`id` is a built-in Python function that returns the memory address of an object. While Python technically allows you to shadow built-in functions by using them as variable names, it is highly discouraged and will trigger warnings in almost all modern linters (like Pylint, Flake8, or Ruff). <br><br>
The standard PEP 8 workaround for avoiding conflict with built-in names is to append a single trailing underscore. Therefore, we name the field `id_` in Python, and use a Pydantic alias to map it to `"id"` for JSON serialization.
</details>